# Google - The Gemma 4 Developer Agent Competition
## Master Pipeline Launcher

This single launcher notebook provides an interactive control panel to run the complete pipeline on Kaggle:
- **Status Check**: Verify environment, models, and data
- **Training**: Launch SFT or DPO fine-tuning
- **Evaluation**: Run local benchmark test and compute Pass@1
- **Packaging**: Compile validated `submission.zip` for submission

In [ ]:
# ---------------------------------------------------------------------------
# Pipeline Import Bootstrap (Compatible with both Local and Kaggle Datasets)
# ---------------------------------------------------------------------------
import os
import sys
from pathlib import Path

# Detect environment and resolve pipeline location
LOCAL_ROOT = Path("..").resolve()
KAGGLE_INPUT = Path("/kaggle/input")

candidate_paths = [
    LOCAL_ROOT,
    LOCAL_ROOT / "pipeline",
    Path("."),
    Path("pipeline")
]

# Check Kaggle dataset mounts
if KAGGLE_INPUT.exists():
    for dataset_dir in KAGGLE_INPUT.glob("*"):
        if (dataset_dir / "pipeline").exists():
            candidate_paths.insert(0, dataset_dir)
        elif (dataset_dir / "main.py").exists() and "pipeline" in dataset_dir.name.lower():
            candidate_paths.insert(0, dataset_dir.parent)

pipeline_loaded = False
for p in candidate_paths:
    if (p / "pipeline" / "main.py").exists() or (p / "pipeline").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        print(f"Successfully loaded pipeline from: {p}")
        pipeline_loaded = True
        break

if not pipeline_loaded:
    print("[WARN] Pipeline folder not auto-detected; defaulting to current sys.path.")

from pipeline.main import PipelineRunner, status, train, evaluate, package
runner = PipelineRunner()
runner.status()

### 1. Workspace Status Check

In [ ]:
runner.status()

### 2. Fine-Tuning Execution (Optional)
Uncomment below to run SFT LoRA fine-tuning.

In [ ]:
# runner.train(mode="sft", config_path="../pipeline/configs/train_lora_sft.yaml")

### 3. Run Benchmark Evaluation

In [ ]:
eval_results = runner.evaluate(tasks_path="../data/val_tasks.jsonl")
print("Evaluation Complete:", eval_results)

### 4. Package & Validate Submission Bundle

In [ ]:
submission_zip = runner.package(
    model_name="gemma-4-12b-it",
    output_zip="../submission.zip"
)
print(f"Submission ready: {submission_zip}")